# 03a — Auto-label SAE features

Owner: Mike Cochran

Pipeline:
1. Load the trained SAE (`mid.sae.load.load_sae`) and the cached activations + metadata (`mid.sae.activations.read_activations`).
2. `top_activating_contexts` — stream activations through the SAE, keep the top-k token positions per feature, and decode a text window around each peak.
3. `label_features` — send those windows to Claude via `mid.analysis.llm.call_anthropic` and parse back `{label, description, confidence}` per feature.
4. Save labels + stats to `../data/sae_labels/` for downstream use (steering, neuron baseline).

In [1]:
import json
import os
import sys
from pathlib import Path

import torch
from dotenv import load_dotenv
from tokenizers import Tokenizer

sys.path.insert(0, "../src")

# Pull ANTHROPIC_API_KEY (and anything else) from a .env at the project root.
# `.env` is gitignored — never commit it.
load_dotenv(Path("..") / ".env")

from mid.analysis.auto_label import label_features, top_activating_contexts
from mid.config import load_configs, load_sae_config
from mid.sae.activations import read_activations
from mid.sae.load import load_sae

## Config

Mirrors the knobs in 02b. `layer` matches the SAE's training layer.

In [2]:
model_type = "decoder"
model_size = "small"
use_split = True  # True -> train.pt; False -> all.pt

# Top-activating-context hyperparams
k = 20            # top-k contexts per feature
window = 16       # tokens on each side of the peak
encode_batch = 8192

# Labeling hyperparams
label_model = "claude-sonnet-4-6"  # sonnet for quality; swap to haiku for cheap smoke tests
max_contexts_per_prompt = 12

model_config_path = f"../configs/model_{model_type}_{model_size}.yaml"
sae_config_path = f"../configs/sae_{model_type}_{model_size}.yaml"
activations_pt_path = (
    f"../data/sae_data/{model_type}_{model_size}/train.pt"
    if use_split
    else f"../data/sae_data/{model_type}_{model_size}/all.pt"
)

model_cfg, _ = load_configs(model_config_path)
sae_cfg = load_sae_config(sae_config_path)

sae_out_dir = f"../data/sae_out/{model_type}_{model_size}/L{sae_cfg.layer}_{sae_cfg.hook_type}"
labels_out_dir = Path(f"../data/sae_labels/{model_type}_{model_size}/L{sae_cfg.layer}_{sae_cfg.hook_type}")
labels_out_dir.mkdir(parents=True, exist_ok=True)

print(sae_cfg)
print("SAE dir:  ", sae_out_dir)
print("Labels ->", labels_out_dir)

SAEConfig(lr=0.0003, batch_size=4096, training_tokens=10000000, context_len=256, l1_coeff=2.0, l1_warmup_steps=500, lr_warmup_steps=100, apply_bias_decay_to_input=True, normalize_activations='expected_average_only_in', dim_input=128, dim_sae=1024, hook_type='stream', layer=3, num_batches_in_buffer=32, store_batch_size_prompts=16, seed=32)
SAE dir:   ../data/sae_out/decoder_small/L3_stream
Labels -> ..\data\sae_labels\decoder_small\L3_stream


## Load SAE, activations, tokenizer

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

sae = load_sae(sae_out_dir, device=device)
activations, metadata, n_tokens, d_in = read_activations(activations_pt_path, layer_num=sae_cfg.layer)
tokenizer = Tokenizer.from_file("../tokenizer_output/tokenizer.json")

assert d_in == sae_cfg.dim_input, f"d_in mismatch: cache={d_in}, sae_cfg={sae_cfg.dim_input}"
assert metadata["seq_len"] == sae_cfg.context_len
print(f"n_tokens={n_tokens:,}  d_in={d_in}  d_sae={sae.cfg.d_sae}")

C:\Users\mjcoc\miniconda3\envs\shakespeare_mech_interp\Lib\site-packages\sae_lens\saes\sae.py:251: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(


n_tokens=1,493,248  d_in=128  d_sae=1024


## Top-activating contexts

One pass over the cached activations. For each feature we keep `k` top-activating token positions and decode a `2*window+1`-token window around each peak (peak wrapped in «»).

In [4]:
top_contexts, stats = top_activating_contexts(
    sae=sae,
    activations=activations,
    metadata=metadata,
    tokenizer=tokenizer,
    k=k,
    window=window,
    batch_size=encode_batch,
    device=device,
)

densities = torch.tensor([s["density"] for s in stats.values()])
dead = int((densities < 1e-5).sum())
ubiq = int((densities > 0.5).sum())
live = len(stats) - dead - ubiq
print(f"features: {len(stats)}  live={live}  dead={dead}  ubiquitous={ubiq}")
print(f"median density of live features: {densities[(densities >= 1e-5) & (densities <= 0.5)].median().item():.4f}")

features: 1024  live=1024  dead=0  ubiquitous=0
median density of live features: 0.0253


### Spot-check: show top contexts for a handful of live features

Eyeball these before spending API budget — if the contexts look like noise, tweak `window` or re-check that `layer` / `hook_type` line up with the SAE.

In [5]:
live_ids = [f for f, s in stats.items() if 1e-5 <= s["density"] <= 0.5]
preview_ids = live_ids[:5]

for fid in preview_ids:
    s = stats[fid]
    print(f"\n--- feature {fid}  density={s['density']:.4f}  max_act={s['max_act']:.2f} ---")
    for e in top_contexts[fid][:5]:
        snippet = e["context_text"].replace("\n", " ")
        print(f"  act={e['activation']:.2f}  {snippet}")


--- feature 0  density=0.0310  max_act=3.63 ---
  act=3.63  onsieur? A word with you.  PAROLLES  Your pleasure,« sir».  LAFEW  Your lord and master did well to make his re
  act=3.53  GENTLEMAN The King's not here.  HELEN  Not here,« sir»?  GENTLEMAN  Not indeed. He hence removed last
  act=3.34  ested groom.  GONERIL  At your choice,« sir».  LEAR I prithee, daughter, do not make me mad.
  act=3.27   To start my quiet.  RODERIGO  Sir,« sir», sir--  BRABANTIO  But thou must needs be
  act=3.24   To most that teach.  PERDITA  Your pardon,« sir». For this I'll blush you thanks.  FLORI

--- feature 1  density=0.0305  max_act=4.09 ---
  act=4.09   stop no more holes but what you should.  S«CH»OOLMASTER  Dii boni! A tink
  act=4.06   my breast against. I shall sleep like a top else.  S«CH»OOLMASTER  Fie, fie, what tedio
  act=4.03  The bitter past, more welcome is the sweet.  EPILO«GU»E  The King's a beggar, now the play is done.
  act=3.95   me!  THESEUS  This way the stag took.  S«CH»OOLM

## Label a small pilot first

Hit 20 live features to validate the prompt and the response parsing before scaling up. Requires `ANTHROPIC_API_KEY` in the environment.

In [6]:
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY before labeling."

pilot_ids = live_ids[:20]
pilot_contexts = {fid: top_contexts[fid] for fid in pilot_ids}
pilot_stats = {fid: stats[fid] for fid in pilot_ids}

pilot_labels = label_features(
    pilot_contexts,
    model=label_model,
    stats=pilot_stats,
    max_contexts=max_contexts_per_prompt,
)

for fid, lab in pilot_labels.items():
    print(f"[{fid:4d}] conf={lab['confidence']:.2f}  {lab['label']!r:40s} - {lab['description']}")

[   0] conf=0.95  'polite address term'                    - Feature fires on 'sir' or 'madam' used as a respectful form of address in dialogue, typically in short responsive utterances or interjections
[   1] conf=0.82  'CH/GH digraph in speaker labels'        - The feature activates on the 'CH' or 'PH' digraph sequence, particularly within capitalized stage direction speaker name tokens such as SCHOOLMASTER, EPILOGUE, PHOEBE, CHAMBERLAIN, CHIEF JUSTICE
[   2] conf=0.97  'Latin/Greek -us suffix'                 - Tokens that are the 'us' ending of Latin or Greek derived proper nouns and words (e.g. Proteus, Cyprus, Hesperus, Coriolanus, Theseus)
[   3] conf=0.45  "mid-word 'v' or syllable boundary"      - Feature activates on letters that are part of 'v'-containing syllables or name fragments split mid-token, particularly 'v' sounds and 'EN' in 'ROSENCRANTZ'
[   4] conf=0.82  "adversative conjunction 'But' starting speaker turn" - The word 'But' appearing at the beginning of a new spe

## Full run over all features

Dead and ubiquitous features are tagged without hitting the API. Each remaining feature makes one API call; at `d_sae=1024`

In [7]:
labels = label_features(
    top_contexts,
    model=label_model,
    stats=stats,
    max_contexts=max_contexts_per_prompt,
)
print(f"labeled {len(labels)} features")

labeled 1024 features


## Persist labels + contexts

Writes two artifacts downstream notebooks can load. Should prevent having to run the feature labeling twice and wasting API credits:
- `labels.json`  — `{feat_id: {label, description, confidence, n_contexts, density, max_act}}`
- `top_contexts.json` — raw top-k contexts per feature (for the steering notebook / manual review)

In [12]:
merged = {
    int(fid): {
        **labels[fid],
        "density": stats[fid]["density"],
        "max_act": stats[fid]["max_act"],
        "mean_act_nonzero": stats[fid]["mean_act_nonzero"],
    }
    for fid in labels
}

with open(labels_out_dir / "labels.json", "w", encoding="utf-8") as f:
    json.dump(merged, f, indent=2, ensure_ascii=False)

print(f"wrote {labels_out_dir / 'labels.json'}")

wrote ..\data\sae_labels\decoder_small\L3_stream\labels.json


## Summarize results

In [13]:
ranked = sorted(
    merged.items(),
    key=lambda kv: (kv[1]["confidence"], kv[1]["max_act"]),
    reverse=True,
)
print(f"{'feat':>5}  {'dens':>6}  {'conf':>4}  label")
for fid, lab in ranked[:25]:
    print(f"{fid:>5}  {lab['density']:>6.4f}  {lab['confidence']:>4.2f}  {lab['label']}")

 feat    dens  conf  label
  885  0.0128  0.99  semicolon
  772  0.0201  0.99  young
  281  0.0196  0.99  imp- prefix
  907  0.0152  0.99  sc- token
  117  0.0237  0.99  tre- prefix
  508  0.0321  0.99  night
  265  0.0273  0.99  acc- prefix
  919  0.0206  0.99  acc- prefix
   52  0.0147  0.99  FALSTAFF speaker tag
  717  0.0198  0.99  thy (second person possessive pronoun)
  608  0.0420  0.99  the word 'one'
  292  0.0264  0.99  or
  229  0.0315  0.99  other
  355  0.0446  0.99  eyes
  436  0.0072  0.98  word starting with 'b'
  669  0.0095  0.98  Exclamatory 'O'
  389  0.0142  0.98  con- prefix token
  739  0.0135  0.98  su- prefix
  775  0.0236  0.98  Go to
  627  0.0145  0.98  en- prefix
  424  0.0214  0.98  never
  352  0.0170  0.98  all (universal quantifier)
   23  0.0208  0.98  bl- onset
  125  0.0196  0.98  great/greatest
  444  0.0172  0.98  the word 'from'
